[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/ROBT407-SVM-Labs/blob/main/student/202_dual_svm.ipynb)

# ROBT 407 · Dual SVMs: Lagrange multipliers, KKT, and support vectors

**Student workbook | Python 3 | Core: 90 minutes | Deepening work + mini-project: 2–3 hours**

Derive the dual constraints; implement its objective and gradient; recover the primal model; verify feasibility, complementary slackness and the duality gap.

Use the cycle **predict → run → inspect → explain**. Write predictions before running experiments. Complete the TODO functions and rerun their checks. Synthetic experiments and bundled scikit-learn datasets are available locally; no dataset downloads are needed. Core activities follow the lecture; deepening labs and mini-projects extend it.

## Lecture alignment and course context
Source: **202_Dual-SVMs.pdf**. PDF pages 3–8: motivation and Lagrangian; 9–15: duality, stationarity and KKT; 16–25: dual QP, support vectors and representation. Page numbers refer to physical PDF pages, starting at 1, rather than the slide footer.

Course context: ROBT 407, Fall 2026; these activities support theoretical reasoning (CLO1), pipeline design and evaluation (CLO2), and Python implementation (CLO3). Textbook context: *Learning From Data*, Yaser S. Abu-Mostafa, Malik Magdon-Ismail, Hsuan-Tien Lin. No textbook excerpt was supplied; these are original explanations and exercises based on the uploaded slides, with no claimed textbook page mapping.

The lecture-specific quiz checks understanding of slide examples; it does not verify whether someone watched a video.

**Notation:** rows are examples; labels are −1 and +1 unless stated otherwise. Bias is separate from w. Margin means one-sided geometric margin unless explicitly called full width.

## Setup
Run cells in order in JupyterLab, Jupyter Notebook, or Google Colab with a Python 3 kernel. Required packages: NumPy, SciPy, Matplotlib, scikit-learn; ipywidgets is optional. Static plots and editable function calls remain available without widgets. Live sliders are opt-in: set ENABLE_WIDGETS=True in Setup and rerun an experiment cell. Static figures are the default for predictable batch execution. Student notebooks intentionally leave implementations incomplete; their checks print reminders until completed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from IPython.display import display
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (7, 5), 'axes.grid': True, 'font.size': 11})
SEED = 407

def boundary(model, X, y, title='', ax=None):
    if ax is None: _, ax = plt.subplots()
    low, high = X.min(axis=0)-.6, X.max(axis=0)+.6
    xx, yy = np.meshgrid(np.linspace(low[0],high[0],160), np.linspace(low[1],high[1],160))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-np.inf,0,np.inf], colors=['#e7edff','#ffe9da'], alpha=.8)
    ax.contour(xx, yy, zz, levels=[-1,0,1], colors='black', linestyles=['--','-','--'])
    ax.scatter(X[:,0],X[:,1],c=y,cmap='coolwarm',edgecolors='black',s=35)
    if hasattr(model,'support_'):
        sv = X[model.support_]
        ax.scatter(sv[:,0],sv[:,1],s=115,facecolors='none',edgecolors='#208050',linewidths=1.5,label='Support vectors')
        ax.legend(loc='best')
    ax.set(xlabel='$x_1$',ylabel='$x_2$',title=title)
    return ax

ENABLE_WIDGETS = False  # Set True and rerun the experiment cells for live sliders.

def optional_slider(fn, name, lo, hi, value, step):
    if not ENABLE_WIDGETS:
        print("To enable live sliders, set ENABLE_WIDGETS = True in Setup; direct function calls also work.")
        return
    # Static experiments remain available if this frontend cannot display widgets.
    try:
        import ipywidgets as widgets
        display(widgets.interactive(fn, **{name: widgets.FloatSlider(min=lo,max=hi,value=value,step=step,continuous_update=False)}))
    except ImportError:
        print('Widgets unavailable; call the experiment function with a new value.')

## 1. From constraints to a dual problem (15 min)
Use $z_n=\Phi(x_n)$; in today's numerical example $z_n=x_n$.
$$L(w,b,\alpha)=\tfrac12\|w\|^2+\sum_n\alpha_n[1-y_n(w^Tz_n+b)],\quad\alpha_n\ge0.$$
Stationarity in $b$ requires $\sum_n\alpha_ny_n=0$; otherwise the infimum over the unbounded bias is $-\infty$. Stationarity in $w$ gives
$$w=\sum_n\alpha_n y_n z_n.$$
Substitution gives the concave maximization
$$\max_{\alpha\ge0}\ \mathbf1^T\alpha-\tfrac12\alpha^TQ\alpha,
\qquad y^T\alpha=0,\quad Q_{nm}=y_ny_mz_n^Tz_m.$$
A minimization solver instead minimizes $\tfrac12\alpha^TQ\alpha-\mathbf1^T\alpha$.

Weak duality gives $D\le P$ for feasible points. For this feasible convex hard-margin SVM, strong duality connects the optima. **Predict:** does the number of dual variables depend on $N$ or the transformed dimension $\tilde d$?

In [ ]:
# Exact four-point example from the linear-SVM lecture.
X = np.array([[0.,0.],[2.,2.],[2.,0.],[3.,0.]])
y = np.array([-1.,-1.,1.,1.])

def solve_primal(X, y):
    d = X.shape[1]
    objective = lambda u: .5 * (u[:d] @ u[:d])
    gradient = lambda u: np.r_[u[:d], 0.]
    A = y[:,None]*np.c_[X,np.ones(len(X))]
    result = minimize(objective, np.zeros(d+1), jac=gradient,
        constraints={'type':'ineq','fun':lambda u:A@u-1,'jac':lambda u:A},
        method='SLSQP',options={'ftol':1e-11,'maxiter':2000})
    if not result.success: raise RuntimeError(result.message)
    w,b = result.x[:d],result.x[d]
    if np.min(y*(X@w+b)) < 1-1e-6: raise RuntimeError('Primal constraints failed')
    return w,b,result.fun

def solve_dual(X, y):
    K = X@X.T
    Q = np.outer(y,y)*K
    result = minimize(lambda a:.5*a@Q@a-a.sum(),np.zeros(len(y)),
        jac=lambda a:Q@a-np.ones(len(y)),bounds=[(0,None)]*len(y),
        constraints={'type':'eq','fun':lambda a:y@a,'jac':lambda a:y},
        method='SLSQP',options={'ftol':1e-12,'maxiter':2000})
    if not result.success: raise RuntimeError(result.message)
    a = result.x
    w = (a*y)@X
    sv = a>1e-7
    b = np.mean(y[sv]-X[sv]@w)
    return a,w,b,-result.fun

a,w,b,dual = solve_dual(X,y)
wp,bp,primal = solve_primal(X,y)
print('alpha:',a,'w:',w,'b:',b)
print('primal objective:',primal,'dual objective:',dual,'gap:',primal-dual)
print('dual equality residual:',y@a)
np.testing.assert_allclose(w,wp,atol=1e-5)
np.testing.assert_allclose(b,bp,atol=1e-5)
assert abs(primal-dual)<1e-6

### Implement A: the minimized dual objective (10 min)
Implement the objective and analytic gradient for an arbitrary symmetric $Q$. Compare the gradient with central finite differences, away from the constraints.

In [ ]:
def dual_value_gradient(alpha,Q):
    # TODO: return (scalar objective, gradient vector).
    return None

In [ ]:
Q = np.outer(y,y)*(X@X.T)
a_test = np.array([.2,.3,.25,.25])
vg = dual_value_gradient(a_test,Q)
if vg is None: print('Complete Implement A, then rerun.')
else:
    value,gradient = vg
    eps=1e-6; eye=np.eye(len(y))
    numerical=np.array([(dual_value_gradient(a_test+eps*e,Q)[0]-dual_value_gradient(a_test-eps*e,Q)[0])/(2*eps) for e in eye])
    np.testing.assert_allclose(gradient,numerical,atol=1e-7)
    np.testing.assert_allclose(value,.5*a_test@Q@a_test-a_test.sum())
    print('Objective and gradient checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## 2. What KKT tells us (10 min)
At an optimum,
$$y_nf(x_n)\ge1,\quad\alpha_n\ge0,\quad y^T\alpha=0,\quad w=\sum_n\alpha_ny_nx_n,$$
$$\alpha_n[1-y_nf(x_n)]=0.$$
Thus $\alpha_n>0$ implies a point is on a margin plane. The converse can fail: a point on a margin plane may have zero multiplier in a degenerate solution. For any positive multiplier,
$$b=y_s-\sum_n\alpha_ny_n x_n^Tx_s.$$
Average over eligible support vectors to reduce numerical error. Use tolerances rather than exact floating-point equality.

In [ ]:
margins = y*(X@w+b)
print('columns: index, alpha, functional margin, complementarity residual')
print(np.c_[np.arange(len(y)),a,margins,a*(1-margins)])
assert np.min(margins)>=1-1e-6
assert np.min(a)>=-1e-8
assert np.max(np.abs(a*(1-margins)))<1e-6

### Implement B: predict using the data representation (10 min)
Implement scores without using the recovered w. Return one score per query. Then compare against the primal scores, including off-training inputs.

In [ ]:
def dual_scores(X_query,X_train,y_train,alpha,b):
    # TODO: use the query-to-training Gram matrix.
    return None

In [ ]:
Xq = np.random.default_rng(SEED).normal(size=(13,2))
scores = dual_scores(Xq,X,y,a,b)
if scores is None: print('Complete Implement B, then rerun.')
else:
    np.testing.assert_allclose(scores,Xq@w+b,atol=1e-7)
    print('Primal / dual prediction agreement passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## 3. Support vectors versus boundary candidates (10 min)
Duplicate the negative point at the origin. Predict whether the optimal boundary changes and whether the individual multipliers must be unique. Investigate the representation, not just the classification accuracy.

In [ ]:
Xd = np.vstack([X,X[0]]); yd = np.r_[y,y[0]]
ad,wd,bd,dd = solve_dual(Xd,yd)
print('duplicated data alpha:',ad)
print('recovered model:',wd,bd,'objective:',dd)
np.testing.assert_allclose(wd,w,atol=1e-5)
np.testing.assert_allclose(bd,b,atol=1e-5)
# A second optimal multiplier assignment: transfer duplicate mass to the original.
alternative = ad.copy(); alternative[0]+=alternative[-1];alternative[-1]=0
np.testing.assert_allclose((alternative*yd)@Xd,wd,atol=1e-6)
print('alternative alpha:',alternative)
print('boundary candidates:',np.flatnonzero(np.isclose(yd*(Xd@wd+bd),1,atol=1e-6)))
print('positive multipliers in alternative:',np.flatnonzero(alternative>1e-7))

## 4. Explain and extend (10 min)
1. Show explicitly that moving multiplier mass between identical same-label points preserves both the objective and the equality constraint.
2. Why can the optimal $w$ be unique while the multipliers are not?
3. The dual has $N$ variables. What computation still depends on $\tilde d$ before we introduce kernels?
4. Estimate the storage for an $N\times N$ float64 Gram matrix when $N=10{,}000$. Why does removing explicit transformed features not remove all scaling costs?

**Submit:** both functions, the KKT residual table, the primal–dual gap and your explanation of the duplicate-point experiment.

## Deepening lab A · A feasible two-variable update (20 min)
Changing only one multiplier usually violates $y^T\alpha=0$. Choose distinct indices $i,j$ and direction $d_i=1$, $d_j=-y_i/y_j$, with other entries zero. Then $y^Td=0$.
For the minimized objective $F(\alpha)=\tfrac12\alpha^TQ\alpha-\mathbf1^T\alpha$,
$$F(\alpha+td)=F(\alpha)+t d^T(Q\alpha-\mathbf1)+\tfrac12t^2d^TQd.$$
When curvature is positive the unconstrained step is $t_*=-d^T(Q\alpha-\mathbf1)/(d^TQd)$. Restrict it to the interval satisfying $\alpha+td\ge0$. This is the basic idea behind feasible pair updates; the exercise below is a single step, not a complete SMO solver.

### Implement C · Exact pairwise step
Compute the feasible interval and clip the optimal step to it. Raise ValueError for zero curvature in this exercise rather than silently divide by zero. Explain why a complete solver would need to handle flat directions and choose useful pairs.

In [ ]:
def pair_step(alpha,Q,y,i,j):
    # TODO: return a new feasible alpha vector with a non-increasing F.
    return None

In [ ]:
Q=np.outer(y,y)*(X@X.T)
updated=pair_step(np.zeros(len(y)),Q,y,0,2)
if updated is None: print('Complete Implement C.')
else:
    np.testing.assert_allclose(updated,[.5,0,.5,0],atol=1e-8)
    assert np.min(updated)>=-1e-10 and abs(y@updated)<1e-10
    F=lambda z:.5*z@Q@z-z.sum()
    assert F(updated)<F(np.zeros(len(y)))
    print('Feasibility and objective improvement passed.')

**Your reasoning:** write your prediction, observations and explanation here.

In [ ]:
# Slice through the feasible dual domain along the optimum direction.
a_star,w_star,b_star,dual_star=solve_dual(X,y)
times=np.linspace(0,2,150)
values=np.array([(t*a_star).sum()-.5*(t*a_star)@Q@(t*a_star) for t in times])
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(times,values);axs[0].axvline(1,color='red',linestyle='--')
axs[0].set(xlabel='Scale t along a feasible ray',ylabel='Dual objective',title='Concave maximization along alpha = t alpha*')
axs[1].bar(np.arange(len(y)),a_star,color=['#4866aa' if yi<0 else '#bf6544' for yi in y])
axs[1].set(xlabel='Training point index',ylabel='Multiplier',title='Only positive multipliers contribute to w')
plt.tight_layout();plt.show()

### Implement D · Diagnose an alleged optimum
Return residuals for primal violation, dual nonnegativity, the equality constraint, stationarity in w and complementary slackness. All should be near zero for a true optimum. Do not use the objective gap alone as a feasibility test.

In [ ]:
def kkt_residuals(X,y,alpha,w,b):
    # TODO: return five nonnegative residuals, in the order above.
    return None

In [ ]:
residual=kkt_residuals(X,y,a_star,w_star,b_star)
if residual is None: print('Complete Implement D.')
else:
    assert np.max(residual)<1e-6
    corrupt=kkt_residuals(X,y,a_star,w_star,b_star+.2)
    assert corrupt.max()>.1
    print('Optimal residuals:',residual,'perturbed bias residuals:',corrupt)
    fig,ax=plt.subplots();idx=np.arange(5)
    ax.bar(idx-.18,residual+.000000001,width=.36,label='Optimum')
    ax.bar(idx+.18,corrupt+.000000001,width=.36,label='Perturbed bias')
    ax.set(yscale='log',xticks=idx,xticklabels=['Primal','Alpha','Equality','w relation','Slackness'],ylabel='Residual + 1e-9',title='A diagnostic is more than training accuracy')
    ax.legend();plt.show()

**Your reasoning:** write your prediction, observations and explanation here.

### Implement E · Delete one observation and refit
Return the recovered (w,b) after removing an indexed training point. Refit the hard-margin dual. Compare deletion of the non-support point with deletion of a support point; draw a conclusion specifically for these data, not a universal rule.

In [ ]:
def refit_without(X,y,index):
    # TODO: remove one row and solve the smaller problem.
    return None

In [ ]:
removed=refit_without(X,y,3)
if removed is None: print('Complete Implement E.')
else:
    np.testing.assert_allclose(removed[0],w_star,atol=1e-5)
    np.testing.assert_allclose(removed[1],b_star,atol=1e-5)
    changes=[]
    for i in range(len(y)):
        wi,bi=refit_without(X,y,i)
        changes.append(np.linalg.norm(np.r_[wi-w_star,bi-b_star]))
    plt.bar(np.arange(len(y)),changes);plt.xlabel('Deleted point index');plt.ylabel('Change in (w,b)');plt.title('Deletion sensitivity in the lecture example');plt.show()
    print('Changes:',changes)

**Your reasoning:** write your prediction, observations and explanation here.

## Real-data mini-project · Wine recognition and binary support-vector reconstruction (45–75 min)
Use classes 0 and 1 from the wine dataset so binary dual coefficients remain directly interpretable.

**Before fitting:** write a hypothesis about the useful kernel or regularization strength, the expected errors and the effect of scaling.

**Search budget:** Tune C only: 5 candidates, 25 fold fits and one refit. These finite-C models use the soft-margin box bound. Gamma is irrelevant for the linear kernel.

Use a stratified 80/20 development/test split. Hyperparameter selection uses **five-fold stratified cross-validation** on development data only. Fit scaling inside each fold through a pipeline. Reserve test labels until selection and refitting are finished. These datasets are bundled with scikit-learn; the loaders need no dataset download. Small benchmark results demonstrate a workflow rather than establish deployment performance.

### Understand the model-selection choices
| Choice | Effect | Applicable kernels |
|---|---|---|
| $C>0$ | Penalty per unit margin violation in the sum-loss objective | Linear, polynomial, RBF |
| $\gamma>0$ | Input scale in similarities | Polynomial, RBF |
| degree | Polynomial power | Polynomial |
| coef0 | Additive polynomial constant | Polynomial |
| kernel | Similarity family and feature geometry | Family comparison |

Distinguish hyperparameter **names**, candidate **values** per name, model **combinations** and total fold **fits**. Conditional grids add branch workloads instead of multiplying irrelevant parameters into every family. Five CV training/validation rotations are performed for each candidate; the winner is refitted on all development data.

SVC handles multiclass tasks using pairwise classifiers. Its packed multiclass coefficient array is not the single binary alpha vector used in earlier derivations.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, balanced_accuracy_score
import pandas as pd
project_data=load_wine()
project_X=project_data.data.copy();project_y=project_data.target.copy()
project_names=np.asarray(project_data.target_names).astype(str)
binary_mask=project_y<2
project_X=project_X[binary_mask];project_y=project_y[binary_mask];project_names=project_names[:2]

X_develop,X_holdout,y_develop,y_holdout=train_test_split(project_X,project_y,test_size=.2,stratify=project_y,random_state=SEED+10)
project_cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED+11)
project_grid={'svc__kernel':['linear'],'svc__C':[.001,.01,.1,1,10]}
project_candidates=len(list(ParameterGrid(project_grid)))
print('Data shape and class counts:',project_X.shape,np.unique(project_y,return_counts=True))
print('Development/test sizes:',len(y_develop),len(y_holdout))
print('Candidates, fold fits, fits including refit:',project_candidates,5*project_candidates,5*project_candidates+1)
assert project_cv.n_splits==5
visits=np.zeros(len(y_develop),dtype=int)
for training,validation in project_cv.split(X_develop,y_develop):
    assert len(np.intersect1d(training,validation))==0
    visits[validation]+=1
assert np.all(visits==1)

### Visual audit before fitting
Inspect development data and class counts. Any exploratory scaling below is separate from the estimator; the CV pipeline still learns its own scaler separately within every fold. For digit images, inspect pixels rather than just flattened vectors.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
labels,counts=np.unique(y_develop,return_counts=True)
axs[0].bar(project_names[labels],counts);axs[0].set(title='Development class counts',ylabel='Observations')
exploreZ=StandardScaler().fit_transform(X_develop)
for label in labels:
    axs[1].scatter(exploreZ[y_develop==label,0],exploreZ[y_develop==label,1],label=project_names[label],alpha=.65)
axs[1].set(xlabel=str(project_data.feature_names[0])+' (scaled)',ylabel=str(project_data.feature_names[1])+' (scaled)',title='Two-feature view; classifier uses all features')
axs[1].legend();plt.tight_layout();plt.show()

### Implement F · Construct a leakage-safe five-fold search
Return an **unfitted** GridSearchCV with a StandardScaler/SVC pipeline named scale and svc. Use the supplied splitter, accuracy scoring, refit=True, return_train_score=True, n_jobs=1 and error_score='raise'.

In [ ]:
def make_project_search(grid,splitter):
    # TODO: construct the pipeline and search.
    return None

In [ ]:
candidate_search=make_project_search(project_grid,project_cv)
if candidate_search is None: print('Complete Implement F to construct your own search.')
else:
    assert isinstance(candidate_search,GridSearchCV)
    assert isinstance(candidate_search.estimator,Pipeline)
    assert isinstance(candidate_search.estimator.named_steps['scale'],StandardScaler)
    assert isinstance(candidate_search.estimator.named_steps['svc'],SVC)
    assert candidate_search.cv.n_splits==5 and candidate_search.scoring=='accuracy'
    assert candidate_search.refit and candidate_search.return_train_score
    assert candidate_search.n_jobs==1 and candidate_search.error_score=='raise'
    assert not hasattr(candidate_search,'cv_results_')
    print('Search configuration checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

### Run the reference workflow, then compare your implementation
This reference is complete so the project can be explored before finishing Implement F. To test your own implementation, replace the reference construction with your function's returned search. The reference supplies the protocol, not your written analysis or conclusions. All tuning uses development data only.

In [ ]:
project_pipeline=Pipeline([('scale',StandardScaler()),('svc',SVC(cache_size=200))])
project_search=GridSearchCV(project_pipeline,project_grid,cv=project_cv,scoring='accuracy',
    refit=True,return_train_score=True,n_jobs=1,error_score='raise')
project_search.fit(X_develop,y_develop)
project_results=pd.DataFrame(project_search.cv_results_)
parameter_columns=[column for column in project_results if column.startswith('param_')]
summary_columns=['rank_test_score',*parameter_columns,'mean_train_score','mean_test_score','std_test_score','mean_fit_time']
display(project_results[summary_columns].sort_values('rank_test_score').head(12))
print('Selected parameters:',project_search.best_params_)
print('Selection mean CV accuracy:',project_search.best_score_)
print('Fold spread is descriptive, not an independent-sample confidence interval.')
baseline_scores=cross_val_score(DummyClassifier(strategy='most_frequent'),X_develop,y_develop,cv=project_cv,scoring='accuracy')
print('Majority baseline mean CV accuracy:',baseline_scores.mean())

In [ ]:
top=project_results.sort_values('rank_test_score').head(10).reset_index(drop=True)
fig,axs=plt.subplots(1,2,figsize=(14,4));x=np.arange(len(top))
axs[0].errorbar(x,top['mean_test_score'],yerr=top['std_test_score'],fmt='o',capsize=4,label='Validation mean +/- fold SD')
axs[0].plot(x,top['mean_train_score'],'s--',label='Mean training accuracy')
axs[0].axhline(baseline_scores.mean(),color='gray',linestyle=':',label='Majority baseline')
axs[0].set(xlabel='Candidate rank (top ten)',ylabel='Accuracy',title='Selection evidence and generalization gap');axs[0].legend(fontsize=8)
axs[1].scatter(project_results['mean_fit_time'],project_results['mean_test_score'],c=project_results['mean_train_score'],cmap='viridis',edgecolors='black')
axs[1].set(xlabel='Mean fit time per fold (seconds)',ylabel='Mean validation accuracy',title='Performance versus fitting cost')
plt.tight_layout();plt.show()

In [ ]:
ordered=project_results.sort_values('param_svc__C')
fig,ax=plt.subplots();Cs=np.array(ordered['param_svc__C'],dtype=float)
ax.errorbar(Cs,ordered['mean_test_score'],yerr=ordered['std_test_score'],marker='o',capsize=4,label='Validation')
ax.plot(Cs,ordered['mean_train_score'],'s--',label='Training')
ax.set(xscale='log',xlabel='C',ylabel='Accuracy',title='Five-fold selection of C');ax.legend();plt.show()

### Reconstruct the refitted binary wine model
Use **scaled** support vectors to reconstruct w, since the pipeline fits the SVM in standardized coordinates. For this binary case, scikit-learn stores signed coefficients alpha*y; class 0 maps to -1 and class 1 to +1. Respect the finite-C box bound.

In [ ]:
best_scale=project_search.best_estimator_.named_steps['scale']
best_svc=project_search.best_estimator_.named_steps['svc']
Zdev=best_scale.transform(X_develop)
w_recovered=best_svc.dual_coef_[0]@Zdev[best_svc.support_]
np.testing.assert_allclose(w_recovered,best_svc.coef_[0],atol=1e-8)
np.testing.assert_allclose(Zdev@w_recovered+best_svc.intercept_[0],project_search.decision_function(X_develop),atol=1e-7)
sv_alpha=np.abs(best_svc.dual_coef_[0]);chosen_C=best_svc.C
assert sv_alpha.max()<=chosen_C+1e-6
print('Support vectors per class:',best_svc.n_support_)
print('Free / bounded (tolerance 1e-5):',np.sum(sv_alpha<chosen_C-1e-5),np.sum(sv_alpha>=chosen_C-1e-5))
fig,ax=plt.subplots(figsize=(10,4));ax.bar(np.arange(len(w_recovered)),w_recovered)
ax.set(xticks=np.arange(len(w_recovered)),xticklabels=project_data.feature_names,ylabel='Weight in scaled coordinates',title='Direction reconstructed from support vectors')
plt.xticks(rotation=70,ha='right');plt.tight_layout();plt.show()

### Freeze the choice, then evaluate once on the held-out test set
Write down the selected configuration and expected failure modes before running this cell. Do not alter the grid based on these results. A new hypothesis after seeing test results requires a fresh evaluation plan; the existing test is no longer untouched.

In [ ]:
holdout_predictions=project_search.predict(X_holdout)
print('Final test accuracy:',accuracy_score(y_holdout,holdout_predictions))
print('Final test balanced accuracy:',balanced_accuracy_score(y_holdout,holdout_predictions))
print(classification_report(y_holdout,holdout_predictions,target_names=project_names,zero_division=0))
fig,ax=plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay.from_predictions(y_holdout,holdout_predictions,display_labels=project_names,cmap='Blues',ax=ax,colorbar=False)
ax.set_title('Final held-out test confusion matrix');plt.xticks(rotation=45,ha='right');plt.tight_layout();plt.show()

### Implement G · Manual five-fold selection
Implement the inner grid search without calling GridSearchCV or cross_val_score. Use ParameterGrid, clone the pipeline for every fold, fit on training indices and score validation indices. Return (best parameter dictionary, list of candidate mean scores). Keep the first candidate in a tie. This is an independent implementation of the selection protocol.

In [ ]:
def manual_grid(estimator,grid,X,y,splitter):
    # TODO: iterate candidates and fold splits; fit fresh clones and average validation scores.
    return None

In [ ]:
# A small two-candidate audit; the main project above uses the full stated grid.
audit_grid={'svc__kernel':['linear'],'svc__C':[.1,1]}
audit=manual_grid(project_pipeline,audit_grid,X_develop,y_develop,project_cv)
if audit is None: print('Complete Implement G for the independent selection challenge.')
else:
    reference=GridSearchCV(project_pipeline,audit_grid,cv=project_cv,scoring='accuracy',n_jobs=1,error_score='raise').fit(X_develop,y_develop)
    best_parameters,mean_scores=audit
    np.testing.assert_allclose(mean_scores,reference.cv_results_['mean_test_score'],atol=1e-12)
    assert best_parameters==reference.best_params_
    print('Manual five-fold grid agrees with GridSearchCV:',best_parameters)

**Your reasoning:** write your prediction, observations and explanation here.

### Submit an evidence-based mini-project report
1. Your pre-experiment hypothesis and evaluation protocol.
2. Tuned parameter names, values, candidate count and five-fold fit count. Account separately for baseline and learning-curve fits.
3. Training versus validation performance, descriptive fold spread, model complexity and fit time. Explain why a small CV difference may be inconclusive.
4. A majority-baseline comparison and per-class error analysis.
5. The frozen choice, final test confusion matrix and limitations.
6. A follow-up experiment with its own fresh evaluation plan.

**Independent challenge:** implement manual five-fold grid search with cloned pipelines, the same folds, the same scoring and development-only model selection. Compare with GridSearchCV, documenting tie-breaking, order and seeds. An instructor reference is supplied separately. An optional further project is nested CV: explain the separate roles of inner selection folds and outer evaluation folds.

Your report is evaluated for reasoning and protocol, not for obtaining the highest test accuracy.

## Lecture understanding quiz (15–20 min)

Answer before discussing with classmates. For each answer, add a one-sentence justification and point to the relevant lecture concept.

**Q1. Which equation comes from differentiating the Lagrangian with respect to b?**  
A: sum(alpha)=0; B: sum(alpha*y)=0; C: b=0; D: sum(y)=0

**Q2. In the hard-margin dual, alpha_n>0 implies what?**  
A: point misclassified; B: point on a margin plane; C: zero bias; D: no constraint

**Q3. Does a point on a margin plane necessarily have alpha_n>0?**  
A: yes; B: no

**Q4. Why can duplicated points admit different optimal alpha vectors?**  
Short answer

**Q5. If there are 1126 positive multipliers among 5566 points, can 1234 points lie on the margin planes?**  
A: yes; B: no

**Q6. How much storage is needed for a 10000-by-10000 float64 Gram matrix?**  
Short answer

**Q7. Why update two multipliers rather than one in a feasible dual step?**  
Short answer

**Q8. Can a small primal-dual objective gap establish optimality if feasibility was not checked?**  
A: yes; B: no

**Q9. Why is the dual objective concave?**  
Short answer

**Q10. Does multiclass SVC store one scalar multiplier per point in one binary vector?**  
A: yes; B: no

In [ ]:
# Record your answers; this cell does not grade or reveal the key.
quiz_answers = {'Q1': '', 'Q2': '', 'Q3': '', 'Q4': '', 'Q5': '', 'Q6': '', 'Q7': '', 'Q8': '', 'Q9': '', 'Q10': ''}
quiz_justifications = {}
display(quiz_answers)

## Exit ticket
In 3–5 sentences: what did your experiment show, how does the equation explain it, and what question remains? Save the notebook with your completed code, plots, quiz responses and explanations.